<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/04_resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04: Clasificación con ResNet-50 — Landslide4Sense

**Proyecto:** Detección de Deslizamientos de Tierra con ML  
**Modelo:** ResNet-50 adaptado a 14 canales multiespectrales  
**Protocolo:** Fine-tuning en dos fases (freeze → unfreeze) + 5-Fold CV

In [ ]:
from google.colab import drive
import os, sys, h5py, subprocess
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

# Instalación de dependencias
packages = ['timm', 'segmentation-models-pytorch', 'h5py', 'tqdm']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Montar Drive
drive.mount('/content/drive')

# Rutas de datos (misma estructura que notebooks 02 y 03)
base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir = img_dirs[0]
    train_mask_dir = train_img_dir.parent / 'mask'
    DATA_ROOT = train_img_dir.parent.parent
    img_list = sorted(list(train_img_dir.glob('*.h5')))
    mask_list = sorted(list(train_mask_dir.glob('*.h5')))
    print(f'✅ {len(img_list)} muestras detectadas en: {DATA_ROOT}')
else:
    print('❌ ERROR: No se encontró TrainData en Drive.')
    print('   Verifica que la carpeta Landslide4Sense esté en MyDrive.')


## 1. Preparar modelo ResNet-50 (14 canales)

In [ ]:
import torch
import timm
import torch.nn as nn

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

# ── Modelo ResNet-50 adaptado a 14 canales ────────────────────────────────
def adapt_first_conv(model, n_channels=14):
    """Cicla pesos ImageNet (3ch) para inicializar 14 canales."""
    old_conv = model.conv1
    new_conv = nn.Conv2d(
        n_channels, old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False
    )
    with torch.no_grad():
        # Ciclar pesos de 3 → 14 canales
        repeated = old_conv.weight.data.repeat(1, (n_channels // 3) + 1, 1, 1)
        new_conv.weight.data = repeated[:, :n_channels, :, :] * (3 / n_channels)
    model.conv1 = new_conv
    return model

class ResNet50Classifier(nn.Module):
    def __init__(self, n_channels=14, pretrained=True):
        super().__init__()
        import torchvision.models as tv
        weights = tv.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = tv.resnet50(weights=weights)
        backbone = adapt_first_conv(backbone, n_channels)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])  # sin FC
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2048, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1)
        )
        self._backbone_frozen = False

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False
        self._backbone_frozen = True

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True
        self._backbone_frozen = False

    def forward(self, x):
        x = self.backbone(x).flatten(1)
        return self.head(x)

model = ResNet50Classifier(n_channels=14, pretrained=True).to(device)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParámetros totales    : {total:,}')
print(f'Parámetros entrenables: {trainable:,}')


## 2. Dataset y DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

class Landslide4SenseDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.augment = augment

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with h5py.File(self.img_paths[idx], 'r') as f:
            img = f['img'][:].astype(np.float32)   # (128, 128, 14)
        with h5py.File(self.mask_paths[idx], 'r') as f:
            mask = f['mask'][:].astype(np.float32) # (128, 128)

        # Z-score por canal
        for c in range(img.shape[2]):
            ch = img[:, :, c]
            std = ch.std() + 1e-8
            img[:, :, c] = (ch - ch.mean()) / std

        label = float(mask.max() > 0)  # clasificación binaria
        img_t = torch.from_numpy(img.transpose(2, 0, 1))  # (14, 128, 128)

        if self.augment:
            if torch.rand(1) > 0.5:
                img_t = torch.flip(img_t, dims=[2])
            if torch.rand(1) > 0.5:
                img_t = torch.flip(img_t, dims=[1])

        return img_t, torch.tensor(label, dtype=torch.float32)

# Etiquetas para stratified split
print('Calculando etiquetas...')
labels = []
for mp in tqdm(mask_list[:200]):  # muestra rápida para verificar
    with h5py.File(mp, 'r') as f:
        labels.append(int(f['mask'][:].max() > 0))
print(f'Positivos (deslizamiento): {sum(labels)} / {len(labels)}')
print(f'Ratio positivos: {sum(labels)/len(labels):.2%}')


## 3. Entrenamiento — 5-Fold Cross Validation

In [ ]:
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import json, time

# ── Configuración ────────────────────────────────────────────────────────────
CFG = {
    'n_folds'      : 5,
    'epochs'       : 20,
    'freeze_epochs': 5,
    'batch_size'   : 32,
    'lr_head'      : 1e-4,
    'lr_backbone'  : 1e-5,
    'pos_weight'   : 0.703,
    'seed'         : 42,
}

torch.manual_seed(CFG['seed'])
output_dir = Path('/content/drive/MyDrive/Landslide4Sense/results/resnet50')
output_dir.mkdir(parents=True, exist_ok=True)

# ── Etiquetas completas ───────────────────────────────────────────────────────
print('Cargando etiquetas completas...')
all_labels = []
for mp in tqdm(mask_list):
    with h5py.File(mp, 'r') as f:
        all_labels.append(int(f['mask'][:].max() > 0))
all_labels = np.array(all_labels)
print(f'Total: {len(all_labels)} | Positivos: {all_labels.sum()} ({all_labels.mean():.2%})')

# ── K-Fold ────────────────────────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
fold_results = []
SEP = '='*50

for fold, (train_idx, val_idx) in enumerate(skf.split(img_list, all_labels)):
    print('\n' + SEP)
    print('FOLD ' + str(fold+1) + '/' + str(CFG['n_folds']))
    print(SEP)

    train_imgs  = [img_list[i] for i in train_idx]
    train_masks = [mask_list[i] for i in train_idx]
    val_imgs    = [img_list[i] for i in val_idx]
    val_masks   = [mask_list[i] for i in val_idx]

    train_ds = Landslide4SenseDataset(train_imgs, train_masks, augment=True)
    val_ds   = Landslide4SenseDataset(val_imgs,   val_masks,   augment=False)
    train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    model = ResNet50Classifier(n_channels=14, pretrained=True).to(device)
    model.freeze_backbone()

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([CFG['pos_weight']]).to(device)
    )
    optimizer = optim.AdamW([
        {'params': model.head.parameters(),     'lr': CFG['lr_head']},
        {'params': model.backbone.parameters(), 'lr': CFG['lr_backbone']},
    ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])

    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}
    best_f1, best_epoch = 0, 0

    for epoch in range(CFG['epochs']):
        if epoch == CFG['freeze_epochs']:
            model.unfreeze_backbone()
            print('  Epoca ' + str(epoch+1) + ': backbone descongelado')

        # Train
        model.train()
        train_loss = 0
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs).squeeze(1), lbls)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_dl)

        # Validación
        model.eval()
        val_loss, all_probs, all_preds, all_true = 0, [], [], []
        with torch.no_grad():
            for imgs, lbls in val_dl:
                imgs, lbls = imgs.to(device), lbls.to(device)
                logits = model(imgs).squeeze(1)
                val_loss += criterion(logits, lbls).item()
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.extend(probs)
                all_preds.extend((probs > 0.5).astype(int))
                all_true.extend(lbls.cpu().numpy())
        val_loss /= len(val_dl)
        f1  = f1_score(all_true, all_preds, zero_division=0)
        auc = roc_auc_score(all_true, all_probs) if len(set(all_true)) > 1 else 0.0

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(f1)
        history['val_auc'].append(auc)
        scheduler.step()

        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch + 1
            torch.save(model.state_dict(), output_dir / f'fold{fold+1}_best.pt')

        print(
            f'  Epoca {epoch+1:02d}/{CFG["epochs"]} | '
            f'loss {train_loss:.4f}/{val_loss:.4f} | '
            f'F1 {f1:.4f} | AUC {auc:.4f}'
        )

    fold_results.append({
        'fold': fold+1,
        'best_f1': best_f1,
        'best_epoch': best_epoch,
        'history': history
    })
    print('  Mejor F1: ' + f'{best_f1:.4f}' + ' (epoca ' + str(best_epoch) + ')')

# Guardar resumen
summary = {
    'model'  : 'resnet50',
    'config' : CFG,
    'folds'  : [{'fold': r['fold'], 'best_f1': r['best_f1'], 'best_epoch': r['best_epoch']} for r in fold_results],
    'mean_f1': float(np.mean([r['best_f1'] for r in fold_results])),
    'std_f1' : float(np.std([r['best_f1']  for r in fold_results])),
}
with open(output_dir / 'kfold_summary.json', 'w') as fout:
    json.dump(summary, fout, indent=2)

print('\n' + SEP)
print('RESNET-50 COMPLETADO')
print(f'F1 medio : {summary["mean_f1"]:.4f} +/- {summary["std_f1"]:.4f}')
print(SEP)


## 4. Visualización de resultados

In [ ]:
fig, axes = plt.subplots(len(fold_results), 2, figsize=(14, 4*len(fold_results)))
if len(fold_results) == 1:
    axes = [axes]

for i, r in enumerate(fold_results):
    h = r['history']
    epochs = range(1, len(h['train_loss']) + 1)

    axes[i][0].plot(epochs, h['train_loss'], label='Train')
    axes[i][0].plot(epochs, h['val_loss'],   label='Val')
    axes[i][0].set_title(f'Fold {r["fold"]} — Loss')
    axes[i][0].legend()
    axes[i][0].set_xlabel('Época')

    axes[i][1].plot(epochs, h['val_f1'],  label='F1',  color='green')
    axes[i][1].plot(epochs, h['val_auc'], label='AUC', color='orange')
    axes[i][1].axvline(r['best_epoch'], color='red', linestyle='--', label=f'Mejor F1={r["best_f1"]:.3f}')
    axes[i][1].set_title(f'Fold {r["fold"]} — Métricas')
    axes[i][1].legend()
    axes[i][1].set_xlabel('Época')

plt.suptitle('ResNet-50 — Historial de Entrenamiento', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en Drive: {output_dir}/training_history.png')


## 5. Resumen

| Fold | Mejor F1 | Época |
|------|----------|-------|
| Se imprime arriba al finalizar el entrenamiento |

Los pesos del mejor modelo de cada fold se guardan automáticamente en Drive:
`MyDrive/Landslide4Sense/results/resnet50/fold{N}_best.pt`

Siguiente paso → `05_efficientnet_b4.ipynb`